In [1]:
# クラスカル法を実装する。

# 図4.38の実装を行う。
distances = [[2, [1, 2]], [6, [1, 3]], [7, [2, 3]], [8, [2, 4]], [4, [3, 4]], [3, [3, 5]], [5, [4, 5]]]
nodeNum = 5


In [2]:
import copy

class Sort:
    # reverseがFalseなら昇順に並べる。
    def __init__(self, data, reverseFlag = False):
        self.sortedData = copy.deepcopy(data)
        if reverseFlag:
            self.checkFunc = self.checkSize
        else:
            self.checkFunc = self.checkReverseSize
        self.sort()

    def checkSize(self, a, b):
        return (a > b)
    
    def checkReverseSize(self, a, b):
        return (a < b)
    
    def pop(self):
        retValue = self.sortedData[0]
        self.sortedData = self.sortedData[1:]
        return retValue

class bubbleSort(Sort):
    def sort(self):
        dataLength = len(self.sortedData)

        # ここのループがデータ数の2乗のオーダ
        for n in range(dataLength):
            for m in range(n, dataLength):
                if not self.checkFunc(self.sortedData[n][0], self.sortedData[m][0]):
                    # データを入れ替える
                    tmp = self.sortedData[n]
                    self.sortedData[n] = self.sortedData[m]
                    self.sortedData[m] = tmp

print("distance : ", distances)
bubble = bubbleSort(distances)
print(bubble.sortedData)
print(bubble.pop())
print(bubble.sortedData)

class quickSort(Sort):
    def sort(self):
        self.sortedData = self.devideData(self.sortedData)

    # ここは平均nlog(n)の計算量。最悪計算量はn^2のオーダ
    def devideData(self, data):
        retValue = []

        if len(data) > 0:
            # 中央値をとるのがよいのかもしれないが、ここでは最初の要素に対して、比較を行う。
            firstElement = data[0]

            div1 = []
            div2 = []
            for i in range(1, len(data)):
                if self.checkFunc(data[i][0], firstElement[0]):
                    div1.append(data[i])
                else:
                    div2.append(data[i])

            retValue = self.devideData(div1) + [firstElement] + self.devideData(div2)

        return retValue

print("distance : ", distances)
quick = quickSort(distances)
print(quick.sortedData)
print(quick.pop())
print(quick.sortedData)


distance :  [[2, [1, 2]], [6, [1, 3]], [7, [2, 3]], [8, [2, 4]], [4, [3, 4]], [3, [3, 5]], [5, [4, 5]]]
[[2, [1, 2]], [3, [3, 5]], [4, [3, 4]], [5, [4, 5]], [6, [1, 3]], [7, [2, 3]], [8, [2, 4]]]
[2, [1, 2]]
[[3, [3, 5]], [4, [3, 4]], [5, [4, 5]], [6, [1, 3]], [7, [2, 3]], [8, [2, 4]]]
distance :  [[2, [1, 2]], [6, [1, 3]], [7, [2, 3]], [8, [2, 4]], [4, [3, 4]], [3, [3, 5]], [5, [4, 5]]]
[[2, [1, 2]], [3, [3, 5]], [4, [3, 4]], [5, [4, 5]], [6, [1, 3]], [7, [2, 3]], [8, [2, 4]]]
[2, [1, 2]]
[[3, [3, 5]], [4, [3, 4]], [5, [4, 5]], [6, [1, 3]], [7, [2, 3]], [8, [2, 4]]]


In [3]:
# 図4.38のようにアルゴリズム4.2のまま実装する。

fs = [i for i in range(nodeNum)]
result = []

# step1 辺の長さを昇順に並び替える
# ここが平均|E|log|V|のオーダ
quick = quickSort(distances)

# step2 |T| = |V| - 1で終了する。
# ここは最悪|E|のオーダ
while len(result) < nodeNum - 1:
    minVal = quick.pop()

    # ここをやるのは|V|のオーダ
    if fs[minVal[1][0] - 1] != fs[minVal[1][1] - 1]:
        result.append(minVal)

        # 別に連結成分の個数を数えなくてもどちらかに寄せればよいのでは？
        # どうせ全体をスキャンしないとfを更新すべきかがわからない。
        # 個数を数えても数えなくても|V|のオーダ。
        # 結果的に|E|=|V|^2のオーダになりそう。
        modifiedIndex = fs[minVal[1][1] - 1]
        for i in range(len(fs)):
            if fs[i] == modifiedIndex:
                fs[i] = fs[minVal[1][0] - 1]

print(result)
    

[[2, [1, 2]], [3, [3, 5]], [4, [3, 4]], [6, [1, 3]]]


In [4]:
# 連結リストと関連しているweighted-union heuristicあたりを使って、少し計算量を減らす。
# https://ja.wikipedia.org/wiki/%E3%82%AF%E3%83%A9%E3%82%B9%E3%82%AB%E3%83%AB%E6%B3%95
# https://ja.wikipedia.org/wiki/%E7%B4%A0%E9%9B%86%E5%90%88%E3%83%87%E3%83%BC%E3%82%BF%E6%A7%8B%E9%80%A0
# http://web.archive.org/web/20181213115442/http://topcoder.g.hatena.ne.jp/iwiwi/20131226/1388062106
# 
# |E| = |V|^2のオーダが|V|log|V|のオーダになるが、本質的にはクイックソートがドミナントになりそうなので、そこまで速くならなさそう。

fs = [i for i in range(nodeNum)]
fsSize = [1 for i in range(nodeNum)]
result = []

# step1 辺の長さを昇順に並び替える
# ここが平均|E|log|V|のオーダ
quick = quickSort(distances)

# step2 |T| = |V| - 1で終了する。
# ここは最悪|E|のオーダ
while len(result) < nodeNum - 1:
    minVal = quick.pop()

    # ここをやるのは|V|のオーダ
    if fs[minVal[1][0] - 1] != fs[minVal[1][1] - 1]:
        result.append(minVal)

        # 本にあるように個数を比較する。
        # それにより|V|log|V|のオーダになりそう。
        if fsSize[minVal[1][0] - 1] < fsSize[minVal[1][1] - 1]:
            modifiedIndex = fs[minVal[1][0] - 1]
            fsSize[minVal[1][1] - 1] += fsSize[minVal[1][0] - 1]
        else:
            modifiedIndex = fs[minVal[1][1] - 1]
            fsSize[minVal[1][0] - 1] += fsSize[minVal[1][1] - 1]

        for i in range(len(fs)):
            if fs[i] == modifiedIndex:
                fs[i] = fs[minVal[1][0] - 1]

print(result)

[[2, [1, 2]], [3, [3, 5]], [4, [3, 4]], [6, [1, 3]]]
